# 09. RQ3: PCAOB Deficiency Severity Classification

**RQ3:** Using the PCAOB's newly released (April 2025) machine-readable inspection
datasets, can a supervised ML classification model accurately predict whether a
deficiency is classified at the more severe Part I.A level versus Part I.B?

**Method:** Logistic regression, benchmarked against a Random Forest classifier.

**Fix applied vs. the original scaffold version:** the original feature list
included `audit_area`, which is structurally populated **only** for Part I.A
records (0% missing) and 100% missing for every Part I.B record: a real
data-leakage bug that produced a suspicious 100% accuracy when first tested.
`audit_area` (and the other Part-I.A-only fields) were removed at the cleaning
stage (see `03_data_cleaning.ipynb`), and `class_weight="balanced"` was added to
both models to properly handle the real ~82/18 class imbalance.

Requires `data/cleaned/audit_disclosure_dataset.csv`, produced by `03_data_cleaning.ipynb`.


In [1]:
!pip install -q pandas scikit-learn || pip install -q pandas scikit-learn --break-system-packages

In [2]:
import os
for _d in ["../data/raw", "../data/cleaned"]:
    os.makedirs(_d, exist_ok=True)

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, confusion_matrix)

# "audit_area" intentionally excluded -- see fix note above.
CATEGORICAL_FEATURES = ["firm_network_category", "standard_cited", "inspection_type", "country"]
NUMERIC_FEATURES = ["inspection_year", "finding_count"]
TARGET = "severity_part"

In [3]:
def build_pipeline(model):
    preprocessor = ColumnTransformer(transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_FEATURES),
    ], remainder="passthrough")
    return Pipeline([("preprocess", preprocessor), ("model", model)])


def evaluate(name, model, X_test, y_test):
    preds = model.predict(X_test)
    probs = model.predict_proba(X_test)[:, 1]
    print(f"\n=== {name} ===")
    print(f"Accuracy:  {accuracy_score(y_test, preds):.3f}")
    print(f"Precision: {precision_score(y_test, preds):.3f}")
    print(f"Recall:    {recall_score(y_test, preds):.3f}")
    print(f"F1:        {f1_score(y_test, preds):.3f}")
    print(f"AUC:       {roc_auc_score(y_test, probs):.3f}")
    print(f"Confusion matrix:\n{confusion_matrix(y_test, preds)}")

## Load the cleaned, leakage-free audit dataset

In [4]:
import urllib.request
AUDIT_PATH = "../data/cleaned/audit_disclosure_dataset.csv"
GITHUB_BASE = "https://raw.githubusercontent.com/daljeetkaurJohar/qm640-governance-analytics/master"
if not os.path.exists(AUDIT_PATH):
    try:
        print("Not found locally -- fetching from GitHub repo...")
        req = urllib.request.Request(f"{GITHUB_BASE}/data/cleaned/audit_disclosure_dataset.csv",
                                      headers={"User-Agent": "qm640-capstone"})
        with urllib.request.urlopen(req, timeout=30) as resp:
            content = resp.read()
        os.makedirs("../data/cleaned", exist_ok=True)
        with open(AUDIT_PATH, "wb") as f:
            f.write(content)
        print(f"Downloaded {len(content)} bytes from GitHub -> {AUDIT_PATH}")
    except Exception as e:
        print(f"GitHub fetch failed ({e}) -- run 03_data_cleaning.ipynb first.")

df = pd.read_csv(AUDIT_PATH)
df = df.dropna(subset=CATEGORICAL_FEATURES + NUMERIC_FEATURES + [TARGET])
print(f"Modeling sample: N = {len(df)} real PCAOB deficiency records")
print(f"Class balance -- Part I.A: {df[TARGET].mean()*100:.1f}%, Part I.B: {(1-df[TARGET].mean())*100:.1f}%")

X = df[CATEGORICAL_FEATURES + NUMERIC_FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

Not found locally -- fetching from GitHub repo...
Downloaded 2181804 bytes from GitHub -> ../data/cleaned/audit_disclosure_dataset.csv
Modeling sample: N = 16704 real PCAOB deficiency records
Class balance -- Part I.A: 82.3%, Part I.B: 17.7%


## Train and evaluate both models

In [5]:
logit_pipeline = build_pipeline(LogisticRegression(max_iter=5000, class_weight="balanced"))
logit_pipeline.fit(X_train, y_train)
evaluate("Logistic Regression (RQ3 baseline)", logit_pipeline, X_test, y_test)


=== Logistic Regression (RQ3 baseline) ===
Accuracy:  0.963
Precision: 0.989
Recall:    0.966
F1:        0.977
AUC:       0.990
Confusion matrix:
[[ 562   30]
 [  93 2656]]


In [6]:
rf_pipeline = build_pipeline(RandomForestClassifier(n_estimators=300, random_state=42, class_weight="balanced"))
rf_pipeline.fit(X_train, y_train)
evaluate("Random Forest (RQ3 benchmark)", rf_pipeline, X_test, y_test)


=== Random Forest (RQ3 benchmark) ===
Accuracy:  0.971
Precision: 0.987
Recall:    0.977
F1:        0.982
AUC:       0.986
Confusion matrix:
[[ 557   35]
 [  63 2686]]
